# E-commerce / Retail Sales Performance Analysis

This notebook runs the reusable analysis pipeline and explores revenue, profit and returned-order trends using Pandas, GroupBy, pivot tables, Matplotlib and Seaborn.

**Before starting:** run `python scripts/download_data.py` and `python scripts/run_analysis.py` from the project root, or run the equivalent cells below.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image, Markdown

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'scripts').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from run_analysis import run

RAW_FILE = PROJECT_ROOT / 'data' / 'raw' / 'raw_superstore.csv'
if not RAW_FILE.exists():
    raise FileNotFoundError('Run `python scripts/download_data.py` from the project root first.')

## 1. Clean and prepare the data

The raw CSV contains three appended sections: order lines, regional managers and returned-order IDs. The pipeline separates these sections, parses dates, standardises numeric fields, handles missing category labels, checks duplicate line IDs and flags returned orders.

In [ ]:
results = run(RAW_FILE)
orders = results['clean']
quality = results['quality']
summaries = results['summaries']
print(f"Clean order lines: {len(orders):,}")
print(f"Distinct orders: {orders['Order ID'].nunique():,}")
print(f"Date range: {orders['Order Date'].min():%Y-%m-%d} to {orders['Order Date'].max():%Y-%m-%d}")
display(pd.Series(quality, name='Value').to_frame())

## 2. KPI overview

Revenue is the sum of `Sales`; profit is the sum of the supplied `Profit` field. Return rate uses distinct returned order IDs divided by distinct orders, avoiding double counting when an order contains several product lines.

In [ ]:
revenue = orders['Sales'].sum()
profit = orders['Profit'].sum()
order_count = orders['Order ID'].nunique()
returned_count = orders.loc[orders['Is Returned'], 'Order ID'].nunique()
kpis = pd.DataFrame({
    'Metric': ['Revenue', 'Profit', 'Profit margin', 'Distinct orders', 'Returned orders', 'Return rate'],
    'Value': [revenue, profit, profit / revenue if revenue else 0, order_count, returned_count, returned_count / order_count if order_count else 0]
})
display(kpis)

## 3. Monthly trend using GroupBy

The monthly table aggregates revenue, profit, order lines, distinct orders and returned orders. It includes zero-activity months inside the dataset's date range so the time series is continuous.

In [ ]:
monthly = summaries['monthly'].copy()
display(monthly.head(12))
peak_month = monthly.loc[monthly['Sales'].idxmax()]
print(f"Peak revenue month: {peak_month['Order Month']:%B %Y} — ${peak_month['Sales']:,.2f}")

## 4. Region and category pivot tables

Pivot tables expose the same measures across two dimensions and help identify areas where revenue and profit move differently.

In [ ]:
profit_by_category_region = pd.pivot_table(
    orders, index='Sub-Category', columns='Region', values='Profit',
    aggfunc='sum', fill_value=0, margins=True, margins_name='Total'
)
display(profit_by_category_region)
display(summaries['region'])
display(summaries['category'])

## 5. Visual review

The pipeline saves seven figures. The chart files can be reused in a presentation or added to the GitHub README.

In [ ]:
for chart in results['charts']:
    print(chart)
    display(Image(filename=str(PROJECT_ROOT / 'outputs' / 'charts' / chart)))

## 6. Business interpretation

Open `reports/insights.md` for the data-generated narrative: peak months, category and sub-category profitability, regional performance, return-rate comparisons and recommended follow-up actions. Read the caveats before presenting conclusions: the sample is fictional and no refund-value field is available.